# 03 – Bee Haven: Wetterdaten und Sonnenstand (N3)

**Ziel:** Außenwetter und Sonnenstand für den Standort (Parameter `STADT`, Koordinaten aus `konfiguration/standorte.csv`) bereitstellen, damit Gold (N4) und die Analyse (N5) das Flugverhalten mit Außentemperatur, Regen, Sonnenschein und Tageszeit (Tag · Dämmerung · Nacht) vergleichen kann.

**Quellen:**
- Wetter: **Bright Sky** (`api.brightsky.dev`) – freie Schnittstelle zu den Open-Data-Messwerten des **Deutschen Wetterdienstes (DWD)**, stündlich, ohne Anmeldung; Vorbild: Kurs-Notebooks 02–04.
- Sonnenstand: Berechnung nach den Formeln der US-Wetter- und Ozeanbehörde **NOAA** („General Solar Position Calculations“), gegengeprüft mit **Open-Meteo**.

**Grundsatz:** Rohantwort unverändert nach `bronze/archive` (Archiv), aufbereitete Daten mit UTC-Zeitstempel nach `silver/cleaned` – wie bei den Sensordaten.

**Quellenangaben und Lizenzen:** Datenbasis: Deutscher Wetterdienst (DWD Open Data, über Bright Sky – https://brightsky.dev/docs/) · Sonnenstand: NOAA, „General Solar Position Calculations“ (https://gml.noaa.gov/grad/solcalc/solareqns.PDF) · Gegenprüfung: Open-Meteo Historical Weather API (https://open-meteo.com/en/docs/historical-weather-api), Lizenz CC BY 4.0.

> **Synapse-Version (Azure Synapse Analytics)** – automatisch erzeugt aus `../03_wetter.ipynb` durch
> `Privat/03_notebooks_ITS-Data/generator/make_synapse_notebooks.py`. **Nicht von Hand ändern**, sondern das lokale
> Notebook ändern und den Generator erneut ausführen.
>
> Unterschiede zur lokalen Version (Vorbild: Kurs-Notebook 04 „weather pipeline synapse ready“):
> - Pfade als **ABFSS-URI (Azure Blob File System)** `abfss://<container>@<konto>.dfs.core.windows.net/...` statt `pathlib.Path`.
> - **Parameterzelle (Parameter Cell)** direkt darunter: eine Pipeline (Notebook-Aktivität) kann die Werte überschreiben.
> - **Kein Kontoschlüssel (Account Key) im Code:** Zugriff über die Identität des Synapse-Arbeitsbereichs (Workspace Identity)
>   bzw. einen verknüpften Dienst (Linked Service). Das Kurs-Notebook trägt den Schlüssel im Klartext ein – Sicherheitsrisiko.
> - Diagramme werden angezeigt; als PNG in den Lake nur bei `BILDER_SPEICHERN = True`.
> - Kosten: Ausführung nur nach Freigabe (Spark-Pool kostet je Minute).

In [ ]:
# --- Parameter (Parameters) – in Synapse als Parameterzelle markiert; Pipeline kann Werte überschreiben ---
STORAGE_ACCOUNT = "mabeehavenlakehouse"   # Speicherkonto (Storage Account, ADLS Gen2) des Lakehouse
CONTAINER = "lakehouse"                   # Container mit bronze/, silver/, gold/
STADT = "schwartau"                       # Standort = Unterordner in bronze/silver/gold und Teil der Dateinamen
LINKED_SERVICE = ""                       # leer = Identität des Arbeitsbereichs; sonst Name des verknüpften Dienstes (Linked Service)
BILDER_SPEICHERN = False                  # Diagramme zusätzlich als PNG nach gold/<stadt>/bilder/ schreiben?
LOKALER_TEST = ""                            # leer = Azure; lokaler Ordner (Spiegel des Containers) = Test ohne Azure

In [ ]:
# --- Hilfsfunktionen für Dateizugriffe: Azure Data Lake (abfss) oder lokaler Testordner ---
import os                                 # lokale Ordner (nur Testmodus)
import fnmatch                            # Dateinamen mit Platzhalter * vergleichen
import tempfile                           # Zwischenablage für PNG-Dateien

AZURE = not LOKALER_TEST                                                   # True = in Synapse gegen den Data Lake arbeiten
WURZEL = (f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"   # ABFSS-Wurzel des Containers (wie Kurs-Notebook 04)
          if AZURE else LOKALER_TEST.rstrip("/") + "/")                    # oder lokaler Testordner
SO = ({"linked_service": LINKED_SERVICE} if LINKED_SERVICE else {}) if AZURE else None   # storage_options für pandas
# Azure: SO nie None – sonst versucht pandas bei Parquet zuerst pyarrow direkt mit abfss:// und bricht ab
# ("Arrow compiled without Azure Blob File System support"); mit {} nimmt pandas den Synapse-Zugang über fsspec.
if AZURE:
    from notebookutils import mssparkutils                                 # Synapse-Werkzeuge: Dateien auflisten, Ordner anlegen, kopieren

def pfad(*teile):
    """Pfad unterhalb der Wurzel zusammensetzen, z. B. pfad("silver", "cleaned") -> abfss://.../silver/cleaned"""
    return WURZEL + "/".join(str(t).strip("/") for t in teile)

def dateien(ordner, muster="*"):
    """Sortierte Liste (voller Pfad, Größe in Byte) aller Dateien in EINEM Ordner, deren Name zum Muster passt."""
    if AZURE:
        try:
            eintraege = [(f.path, f.size) for f in mssparkutils.fs.ls(ordner) if f.isFile and fnmatch.fnmatch(f.name, muster)]
        except Exception:                                                  # Ordner existiert (noch) nicht
            eintraege = []
    else:
        eintraege = [(os.path.join(ordner, n), os.path.getsize(os.path.join(ordner, n)))
                     for n in (os.listdir(ordner) if os.path.isdir(ordner) else []) if fnmatch.fnmatch(n, muster)]
    return sorted(eintraege)                                               # sortiert nach Name -> letzte = neueste Version (Zeitstempel)

def neueste(ordner, muster):
    """Pfad der jüngsten Datei (Name enthält UTC-Zeitstempel) – Fehler, wenn keine vorhanden."""
    treffer = dateien(ordner, muster)
    if not treffer:
        raise FileNotFoundError(f"keine Datei {muster} in {ordner}")
    return treffer[-1][0]

def ordner_anlegen(ordner):
    """Zielordner anlegen, falls nicht vorhanden (in ADLS Gen2 unschädlich, wenn schon da)."""
    mssparkutils.fs.mkdirs(ordner) if AZURE else os.makedirs(ordner, exist_ok=True)

def archivieren(datei, neuer_name=None):
    """Datei in das _archiv ihres eigenen Ordners verschieben (Regel: jeder Ordner hat ein _archiv) – nichts wird gelöscht."""
    ordner, name = datei.rsplit("/", 1)
    ziel = f"{ordner}/_archiv/{neuer_name or name}"
    ordner_anlegen(f"{ordner}/_archiv")
    mssparkutils.fs.mv(datei, ziel) if AZURE else os.replace(datei, ziel)

def groesse_mb(datei):
    """Dateigröße in Megabyte (zur Kontrolle nach dem Speichern)."""
    ordner, name = datei.rsplit("/", 1)
    return next((g for p, g in dateien(ordner, name)), 0) / 1_000_000

def bild_speichern(fig, name, **optionen):
    """Diagramm optional als PNG in den Lake schreiben (Parameter BILDER_SPEICHERN)."""
    if not BILDER_SPEICHERN:
        return
    ziel_ordner = pfad("gold", STADT, "bilder")
    ordner_anlegen(ziel_ordner)
    lokal = os.path.join(tempfile.gettempdir(), name)                      # erst lokal auf dem Knoten speichern ...
    fig.savefig(lokal, **optionen)
    if AZURE:
        mssparkutils.fs.cp("file:" + lokal, ziel_ordner + "/" + name)       # ... dann in den Data Lake kopieren
    else:
        os.replace(lokal, os.path.join(ziel_ordner, name))

print("Wurzel:", WURZEL, "| Azure:", AZURE)

## 0. Bibliotheken, Pfade und Standort

**Kontrollwert:** keine Ausgabe, keine Fehlermeldung

In [ ]:
# --- Bibliotheken laden ---
import time                                   # kurze Pause zwischen Wiederholungsversuchen
from datetime import datetime, timezone       # UTC-Zeitstempel für Dateinamen
import numpy as np                            # Winkelfunktionen für den Sonnenstand
import pandas as pd                           # Tabellen
import requests                               # Abruf der Wetter-Schnittstelle (HTTP)
import matplotlib.pyplot as plt               # Diagramme
import matplotlib.dates as mdates             # Datumsformat der x-Achse

# --- Pfade (Struktur im Container lakehouse) ---
ZWISCHENSTAND = pfad("silver", "zwischenstand", STADT)                   # Ergebnis aus N1 (für den Zeitraum)
BRONZE_WETTER = pfad("bronze", "archive", STADT, "wetter")               # Rohantwort (Archiv, nie überschreiben)
CLEANED = pfad("silver", "cleaned", STADT)                               # aufbereitete Daten

# --- Standortdaten aus der Konfiguration (Koordinaten, Zeitzone) ---
standorte = pd.read_csv(pfad("konfiguration", "standorte.csv"), index_col="stadt", storage_options=SO)
if STADT not in standorte.index:                          # unbekannte Stadt -> sofort abbrechen statt mit falschen Daten weiterrechnen
    raise ValueError(f"Standort '{STADT}' fehlt in konfiguration/standorte.csv – Zeile mit Koordinaten und Zeitzone ergänzen")
ORT = standorte.loc[STADT]                                # eine Zeile: name, breite, laenge, zeitzone, quelle
ZEITZONE = ORT["zeitzone"]                                # Ortszeit der Rohdaten (Deutschland: Europe/Berlin)

# --- Sensor-Regeln je Standort (Einheit der Rohdaten, gültiger Bereich) und bekannte Ausfallzeiträume ---
sensoren_cfg = pd.read_csv(pfad("konfiguration", "sensoren.csv"), storage_options=SO)          # eine Zeile je Stadt und Sensor
REGELN = sensoren_cfg[sensoren_cfg["stadt"] == STADT].set_index("sensor")              # Regeln dieser Stadt: einheit_roh, min, max
ausschluesse = pd.read_csv(pfad("konfiguration", "ausschluesse.csv"), storage_options=SO)      # bekannte Ausfallzeiträume
AUSSCHLUESSE = ausschluesse[ausschluesse["stadt"] == STADT].copy()                     # nur Zeiträume dieser Stadt
AUSSCHLUESSE[["von_utc", "bis_utc"]] = AUSSCHLUESSE[["von_utc", "bis_utc"]].apply(pd.to_datetime, utc=True)  # Text -> UTC-Zeit
BREITE, LAENGE = float(ORT["breite"]), float(ORT["laenge"])   # Breitengrad (Nord), Längengrad (Ost) des Bienenstocks

**Ergebnis:**

Bibliotheken, Pfade und Standort ohne Fehler geladen. Die Koordinaten stammen aus der Standort-Tabelle – eine fehlende Stadt führt zu einer klaren Fehlermeldung statt zu falschem Wetter.

## 1. Zeitraum der Sensordaten bestimmen

**Warum?** Wetter wird genau für den Zeitraum gebraucht, in dem der Bienenstock gemessen hat – nicht mehr (Datenmenge, Kosten) und nicht weniger.

**Kontrollwert (Zeitraum aus N1):** 01.01.2017 bis 31.05.2019 (UTC)

In [ ]:
# --- Frühesten und spätesten Zeitpunkt über alle 4 Sensoren suchen ---
zeiten = [pd.read_parquet(f"{ZWISCHENSTAND}/{s}_{STADT}_n1.parquet", columns=["zeit_utc"], storage_options=SO)["zeit_utc"]  # nur die Zeitspalte laden
          for s in ["flow", "humidity", "temperature", "weight"]]
start = min(z.min() for z in zeiten)                   # frühester Messzeitpunkt
ende = max(z.max() for z in zeiten)                    # spätester Messzeitpunkt
print("Zeitraum (UTC):", start, "bis", ende)

**Ergebnis:**

Zeitraum 01.01.2017 12:00 bis 31.05.2019 12:15 UTC – Wetter wird für 01.01.2017 bis 01.06.2019 abgerufen (ganze Tage).

## 2. Wetter abrufen und als Rohdaten archivieren (Bronze)

**Wie?** Eine Anfrage an Bright Sky mit Koordinaten und Zeitraum; Einheiten `dwd` (°C, mm, Minuten, km/h). Bei Fehlern bis zu 3 Versuche (wie im Kurs-Notebook).

**Warum archivieren?** Die Rohantwort bleibt unverändert erhalten – wenn die Aufbereitung falsch war, kann sie ohne neuen Abruf wiederholt werden.

**Eigene Erwartung (Rechnung):** 881 Tage × 24 Stunden ≈ 21.100 Stundenwerte, dazu eine Liste der Wetterstationen

In [ ]:
# --- Wetterdaten von Bright Sky (DWD) abrufen ---
def wetter_abrufen(breite, laenge, von, bis, versuche=4):
    """Stündliches Wetter als JSON holen; bei Fehlern mehrfach versuchen."""
    parameter = {"lat": breite, "lon": laenge,           # Standort
                 "date": von, "last_date": bis,          # Zeitraum (ganze Tage)
                 "tz": "UTC", "units": "dwd",            # Zeiten in UTC, Einheiten des DWD
                 "max_dist": 50_000}                     # Stationen im Umkreis von 50 km
    fehler = None
    for versuch in range(1, versuche + 1):               # mehrere Versuche (kurze Störungen überbrücken)
        try:
            antwort = requests.get("https://api.brightsky.dev/weather", params=parameter, timeout=120)
            if antwort.status_code == 200:               # 200 = erfolgreich
                return antwort.json()                    # leere/kaputte Antwort -> Fehler -> nächster Versuch
            fehler = f"Status {antwort.status_code}"
        except (requests.RequestException, ValueError) as e:   # Netzwerkfehler oder ungültiges JSON
            fehler = repr(e)
        time.sleep(5 * versuch)                          # 5, 10, 15 … Sekunden warten, dann erneut versuchen
    raise RuntimeError(f"Bright Sky nach {versuche} Versuchen nicht erreichbar ({fehler})")

roh = wetter_abrufen(BREITE, LAENGE, start.strftime("%Y-%m-%d"), (ende + pd.Timedelta(days=1)).strftime("%Y-%m-%d"))
wetter_roh = pd.json_normalize(roh["weather"])           # Liste der Stundenwerte → Tabelle
stationen = pd.json_normalize(roh["sources"])            # Liste der Wetterstationen → Tabelle

# --- Rohdaten unverändert archivieren (Parquet, mit UTC-Zeitstempel) ---
stempel = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")        # Zeitpunkt des Abrufs
ordner_anlegen(BRONZE_WETTER)                         # Ordner anlegen, falls nötig
wetter_roh.to_parquet(f"{BRONZE_WETTER}/wetter_roh__{stempel}_UTC.parquet", index=False, storage_options=SO)
stationen.to_parquet(f"{BRONZE_WETTER}/stationen_roh__{stempel}_UTC.parquet", index=False, storage_options=SO)
print(f"{len(wetter_roh):,} Stundenwerte · {wetter_roh.shape[1]} Spalten · {len(stationen)} Stationen")

**Ergebnis:**

21.145 Stundenwerte mit 32 Spalten und 6 Stationen abgerufen und in `bronze/archive/schwartau/wetter/` archiviert.

## 3. Welche Station liefert welchen Messwert?

**Warum?** Bright Sky nimmt die nächstgelegene Station und ergänzt fehlende Messgrößen automatisch aus Ersatzstationen (`fallback`). Für die Deutung ist wichtig, **wo** z. B. die Außentemperatur gemessen wurde.

**Erwartung laut Schnittstellenbeschreibung:** Bright Sky wählt die nächstgelegene DWD-Station und ergänzt fehlende Messgrößen aus Ersatzstationen (Felder `fallback_source_ids`, OpenAPI-Beschreibung https://api.brightsky.dev/openapi.json) – also Hauptstation in der Nähe, einzelne Messgrößen von Nachbarstationen.

In [ ]:
# --- Stationen mit Entfernung ---
print(stationen[["id", "station_name", "distance"]].assign(km=lambda t: (t["distance"] / 1000).round(1)).drop(columns="distance").to_string(index=False))

# --- Je Messgröße: Anteil der Stunden je Station ---
name = stationen.drop_duplicates("id").set_index("id")["station_name"]  # Station-ID → Name
herkunft = {}
for g in ["temperature", "precipitation", "sunshine", "wind_speed", "relative_humidity", "cloud_cover",
          "pressure_msl", "dew_point", "wind_gust_speed", "solar", "condition"]:
    spalte = f"fallback_source_ids.{g}"                                   # Ersatzstation für diese Messgröße (falls vorhanden)
    station = wetter_roh[spalte].fillna(wetter_roh["source_id"]) if spalte in wetter_roh else wetter_roh["source_id"]
    herkunft[g] = station.map(name).value_counts(normalize=True).mul(100).round(1)  # Anteil in %
pd.DataFrame(herkunft).fillna(0)                                          # Tabelle: Station × Messgröße (in %)

**Ergebnis (Befund W1):**

Die Hauptstation **Bad Schwartau-Groß Parin (2,1 km)** liefert den **Niederschlag** (98,6 %) und einen Teil des Sonnenscheins. Temperatur, Wind, Luftfeuchte und Bewölkung stammen zu ca. 99 % von **Lübeck-Blankensee (13 km)**, vereinzelt von Pelzerhaken (22 km); Sonnenschein zu 75 % aus Lübeck-Blankensee und 25 % aus Bad Schwartau. Luftdruck, Taupunkt, Windböen und Sonnenstrahlung kommen zu über 99 % aus Lübeck-Blankensee; die Wetterlage (`condition`) zu 88 % aus Bad Schwartau.

→ „Außentemperatur“ heißt in dieser Analyse: gemessen in Lübeck-Blankensee. Für Tagesverläufe ausreichend; kleinräumige Unterschiede (z. B. Sonne direkt am Stock) sind nicht erfasst.

## 4. Datenqualität prüfen

**Prüfungen:** Zeitabstände (erwartet genau 1 Stunde), fehlende Werte je Messgröße, Verteilung der fehlenden Sonnenscheinwerte über die Uhrzeit.

**Eigene Erwartung:** lückenlose Stunden; einzelne fehlende Werte

In [ ]:
# --- Zeitabstände ---
wetter_roh["zeit_utc"] = pd.to_datetime(wetter_roh["timestamp"], utc=True)   # Text → UTC-Zeit
print("Zeitabstände:", wetter_roh["zeit_utc"].diff().value_counts().to_dict())

# --- Fehlende Werte je Messgröße ---
groessen = ["temperature", "precipitation", "sunshine", "wind_speed", "relative_humidity", "cloud_cover", "solar",
            "pressure_msl", "dew_point", "wind_direction", "wind_gust_speed", "wind_gust_direction", "condition"]
print(wetter_roh[groessen].isna().sum().rename("fehlend").to_frame().T.to_string())

# --- Fehlender Sonnenschein: zu welcher Uhrzeit (Ortszeit)? ---
stunde = wetter_roh["zeit_utc"].dt.tz_convert(ZEITZONE).dt.hour
wetter_roh["sunshine"].isna().groupby(stunde).mean().mul(100).round(0).rename("fehlend_%").rename_axis("stunde_ortszeit").to_frame().T

**Ergebnis (Befund W2):**

Stundenwerte lückenlos (21.144 Abstände à 1 Stunde). Fehlend: Sonnenschein 5.287 · Bewölkung 14 · Wetterlage 393 · übrige 0 (auch Luftdruck, Taupunkt, Wind, Böen, Sonnenstrahlung vollständig).

Alle fehlenden Sonnenscheinwerte liegen nachts (23–3 Uhr immer, 22 und 4 Uhr teilweise – je nach Jahreszeit) → die Station meldet nachts keinen Wert statt „0 Minuten“. Regel für N4 Gold und N5: Sonnenschein **nachts = 0** (Sonne unter dem Horizont), tagsüber fehlend bleibt fehlend.

## 5. Sonnenstand berechnen: Tag · Dämmerung · Nacht

**Wie?** Für jeden Zeitpunkt die **Sonnenhöhe** (Winkel über dem Horizont) nach den NOAA-Formeln berechnen:
- **Tag:** Sonnenhöhe > −0,833° (Sonnenaufgang/-untergang laut Definition, inkl. Lichtbrechung)
- **Dämmerung:** −6° bis −0,833° (bürgerliche Dämmerung – es ist noch hell genug für Aktivität im Freien)
- **Nacht:** unter −6°

**Warum?** Sonnenauf- und -untergang verschieben sich über das Jahr (Juni ca. 21:50 Uhr, Dezember ca. 15:55 Uhr) – feste Uhrzeiten für „Tag“ und „Nacht“ wären falsch.

**Erwartung laut astronomischer Näherung:** In ca. 54° Nord dauert der längste Tag (Sommersonnenwende) rund 17 Stunden, der kürzeste (Wintersonnenwende) rund 7½ Stunden; Sonnenuntergang im Sommer nach 21 Uhr, im Winter vor 16 Uhr (Ortszeit).

In [ ]:
# --- Sonnenhöhe nach NOAA ("General Solar Position Calculations") ---
def sonnenhoehe(zeit_utc, breite=BREITE, laenge=LAENGE):
    """Sonnenhöhe in Grad für UTC-Zeitpunkte (Genauigkeit ca. ±2 Minuten bei Auf-/Untergang)."""
    tag = zeit_utc.dt.dayofyear.to_numpy()                                # Tag im Jahr (1–366)
    std = (zeit_utc.dt.hour + zeit_utc.dt.minute / 60).to_numpy()         # Uhrzeit UTC als Dezimalstunde
    g = 2 * np.pi / 365 * (tag - 1 + (std - 12) / 24)                     # Jahreswinkel (Bogenmaß)
    zeitgl = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                       - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))   # Zeitgleichung in Minuten
    dekl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))  # Deklination der Sonne (Bogenmaß)
    ortszeit_sonne = std * 60 + zeitgl + 4 * laenge                       # wahre Sonnenzeit in Minuten
    stundenwinkel = np.radians(ortszeit_sonne / 4 - 180)                  # Stundenwinkel (Bogenmaß)
    b = np.radians(breite)                                                # Breitengrad (Bogenmaß)
    return np.degrees(np.arcsin(np.sin(b) * np.sin(dekl) + np.cos(b) * np.cos(dekl) * np.cos(stundenwinkel)))

def tageszeit(hoehe):
    """Sonnenhöhe → Tag / Dämmerung / Nacht."""
    return np.select([hoehe > -0.833, hoehe > -6], ["Tag", "Dämmerung"], "Nacht")

# --- Sonnenzeiten je Tag aus einer Minutenreihe ableiten ---
minuten = pd.Series(pd.date_range(start.floor("D"), ende.ceil("D"), freq="min", tz="UTC"))  # jede Minute des Zeitraums
m = pd.DataFrame({"zeit_utc": minuten, "hoehe": sonnenhoehe(minuten)})
m["datum"] = m["zeit_utc"].dt.tz_convert(ZEITZONE).dt.date                          # Kalendertag (Ortszeit)
tag = m[m["hoehe"] > -0.833].groupby("datum")["zeit_utc"]                                  # Minuten mit Sonne über dem Horizont
hell = m[m["hoehe"] > -6].groupby("datum")["zeit_utc"]                                      # Minuten inkl. Dämmerung
sonne = pd.DataFrame({"daemmerung_beginn_utc": hell.min(), "sonnenaufgang_utc": tag.min(),
                      "sonnenuntergang_utc": tag.max(), "daemmerung_ende_utc": hell.max()})
sonne["tageslaenge_h"] = ((sonne["sonnenuntergang_utc"] - sonne["sonnenaufgang_utc"]).dt.total_seconds() / 3600).round(2)
sonne = sonne.reset_index()

# --- Kontrolle an Stichtagen (Ortszeit) ---
zeige = sonne.set_index("datum").loc[[pd.Timestamp(d).date() for d in ["2018-03-20", "2018-06-21", "2018-09-23", "2018-12-21"]]]
zeige[["sonnenaufgang_utc", "sonnenuntergang_utc"]].apply(lambda s: s.dt.tz_convert(ZEITZONE).dt.strftime("%H:%M")) \
    .set_axis(["sonnenaufgang_ortszeit", "sonnenuntergang_ortszeit"], axis=1).assign(tageslaenge_h=zeige["tageslaenge_h"])

**Ergebnis:**

Sonnenzeiten für jeden Tag berechnet (Aufgang, Untergang, Beginn/Ende der Dämmerung, Tageslänge). Stichtage in Ortszeit: 21.06.2018 04:45–21:52 Uhr (17,1 h) · 21.12.2018 08:34–15:56 Uhr (7,4 h) – wie erwartet.

## 6. Gegenprüfung mit Open-Meteo

**Warum?** Eine selbst berechnete Größe wird mit einer unabhängigen Quelle verglichen, bevor sie in die Analyse geht (Grundsatz: prüfen und gegenprüfen).

**Eigene Erwartung:** Abweichung nur wenige Minuten (beide Verfahren beruhen auf ähnlichen Sonnenstandsformeln; Open-Meteo rundet auf Minuten)

In [ ]:
# --- Sonnenauf-/-untergang von Open-Meteo (historisches Archiv) holen ---
# Gegenprüfung ist eine Kontrolle, kein Pflichtschritt: bei einer Störung der Schnittstelle nur Hinweis, Pipeline läuft weiter
try:
    om = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=60, params={
        "latitude": BREITE, "longitude": LAENGE,                              # gleicher Standort
        "start_date": start.strftime("%Y-%m-%d"), "end_date": ende.strftime("%Y-%m-%d"),
        "daily": "sunrise,sunset", "timezone": "GMT"}).json()["daily"]        # Tageswerte in UTC
except (requests.RequestException, ValueError, KeyError) as e:
    print("Hinweis: Open-Meteo nicht erreichbar – Gegenprüfung übersprungen:", repr(e))
    om = {"time": [], "sunrise": [], "sunset": []}                           # leere Tabelle -> Vergleich ohne Zeilen
om = pd.DataFrame(om).assign(datum=lambda t: pd.to_datetime(t["time"]).dt.date).set_index("datum")

# --- Abweichung in Minuten ---
vergleich = sonne.set_index("datum").join(om, how="inner")
abw_auf = (vergleich["sonnenaufgang_utc"].dt.tz_localize(None) - pd.to_datetime(vergleich["sunrise"])).dt.total_seconds() / 60
abw_unter = (vergleich["sonnenuntergang_utc"].dt.tz_localize(None) - pd.to_datetime(vergleich["sunset"])).dt.total_seconds() / 60
pd.DataFrame({"mittel_min": [abw_auf.abs().mean(), abw_unter.abs().mean()],
              "max_min": [abw_auf.abs().max(), abw_unter.abs().max()]},
             index=["Sonnenaufgang", "Sonnenuntergang"]).round(1)

**Ergebnis:**

Abweichung zu Open-Meteo im Mittel ca. 2 Minuten, höchstens 5 Minuten → für die Einteilung Tag · Dämmerung · Nacht ausreichend genau.

## 7. Wetter aufbereiten (Silver)

**Was?** Benötigte Messgrößen (Umfang wie Kurs-Notebook 02: zusätzlich Luftdruck, Taupunkt, Windrichtung, Windböen, Sonnenstrahlung, Wetterlage), verständliche Spaltennamen mit Einheit (Einheiten `dwd` laut Bright-Sky-Schnittstellenbeschreibung), Stationen je Messgröße, Sonnenhöhe und Tageszeit je Stunde. Sonnenschein nachts = 0 (Befund W2).

**Kontrollwert (Ergebnis Kap. 2):** 21.145 Zeilen, keine doppelten Zeiten

In [ ]:
# --- Spalten auswählen und verständlich benennen (Einheit im Namen) ---
w = pd.DataFrame({
    "zeit_utc": wetter_roh["zeit_utc"],
    "aussentemperatur_c": wetter_roh["temperature"],          # Lufttemperatur 2 m, °C
    "niederschlag_mm": wetter_roh["precipitation"],           # Niederschlag der letzten Stunde, mm
    "sonnenschein_min": wetter_roh["sunshine"],               # Sonnenscheindauer der letzten Stunde, Minuten
    "wind_kmh": wetter_roh["wind_speed"],                     # Windgeschwindigkeit, km/h
    "luftfeuchte_aussen_pct": wetter_roh["relative_humidity"],# relative Luftfeuchte, %
    "bewoelkung_pct": wetter_roh["cloud_cover"],              # Bewölkung, %
    "luftdruck_hpa": wetter_roh["pressure_msl"],              # Luftdruck auf Meereshöhe reduziert, hPa
    "taupunkt_c": wetter_roh["dew_point"],                    # Taupunkt 2 m, °C
    "windrichtung_grad": wetter_roh["wind_direction"],        # mittlere Windrichtung der letzten Stunde, Grad (0 = Nord)
    "windboeen_kmh": wetter_roh["wind_gust_speed"],           # stärkste Böe der letzten Stunde, km/h
    "windboeen_richtung_grad": wetter_roh["wind_gust_direction"],  # Richtung der stärksten Böe, Grad
    "sonnenstrahlung_kwh_m2": wetter_roh["solar"],            # Sonneneinstrahlung der letzten Stunde, kWh/m²
    "wetterlage": wetter_roh["condition"].map({"dry": "trocken", "fog": "Nebel", "rain": "Regen", "sleet": "Schneeregen",
                                               "snow": "Schnee", "hail": "Hagel", "thunderstorm": "Gewitter"}),  # abgeleitet (best effort)
})

def station(groesse):
    """Name der Station, die diese Messgröße geliefert hat (Ersatzstation, sonst Hauptstation)."""
    spalte = f"fallback_source_ids.{groesse}"
    nummer = wetter_roh[spalte].fillna(wetter_roh["source_id"]) if spalte in wetter_roh else wetter_roh["source_id"]
    return nummer.map(name)                                   # Stationsname statt Nummer

w["station_temperatur"] = station("temperature")              # Befund W1: Stationen je Messgröße festhalten
w["station_niederschlag"] = station("precipitation")
w["station_sonnenschein"] = station("sunshine")

# --- Sonnenstand je Stunde ---
w["sonnenhoehe_grad"] = sonnenhoehe(w["zeit_utc"]).round(2)
w["tageszeit"] = tageszeit(w["sonnenhoehe_grad"])

# --- Regel W2: Sonnenschein nachts = 0 (Sonne unter dem Horizont) ---
nachts_leer = w["sonnenschein_min"].isna() & (w["sonnenhoehe_grad"] <= -0.833)
w.loc[nachts_leer, "sonnenschein_min"] = 0.0
w["qualitaet"] = np.where(w[["aussentemperatur_c", "niederschlag_mm", "sonnenschein_min"]].isna().any(axis=1), "fehlend", "ok")

print(f"{len(w):,} Zeilen · doppelte Zeiten: {w['zeit_utc'].duplicated().sum()} · Sonnenschein nachts auf 0 gesetzt: {nachts_leer.sum():,}")
print(w["tageszeit"].value_counts().to_dict(), "|", w["qualitaet"].value_counts().to_dict())
w.head(3)

**Ergebnis:**

21.145 Stundenzeilen ohne doppelte Zeiten. Alle 5.287 fehlenden Sonnenscheinwerte lagen bei Sonne unter dem Horizont und wurden auf 0 gesetzt (Regel W2); danach alle Zeilen `qualitaet = ok`. Verteilung: 10.768 Stunden Tag · 1.219 Dämmerung · 9.158 Nacht.

## 8. Plausibilität visualisieren

**Was?** (1) Außentemperatur im Tagesmittel über den gesamten Zeitraum · (2) Sonnenauf- und -untergang in Ortszeit über das Jahr.

**Eigene Erwartung:** Sommer ca. 15–25 °C, Winter um 0 °C; Sonnenuntergang zwischen ca. 16 und 22 Uhr

In [ ]:
# --- Grafik: Außentemperatur und Sonnenzeiten ---
fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))                          # zwei Diagramme nebeneinander

tagesmittel = w.set_index("zeit_utc")["aussentemperatur_c"].resample("D").mean()   # Tagesmittel
ax[0].plot(tagesmittel.index, tagesmittel, color="#8c564b", lw=0.8)   # Kategoriefarbe (Farbregel: Rot nur für Bewertungen)
ax[0].axhline(0, color="grey", lw=0.5)                                   # Nulllinie
ax[0].set(title=f"Außentemperatur (Tagesmittel, {w['station_temperatur'].mode()[0]})", ylabel="°C")

lokal = lambda s: s.dt.tz_convert(ZEITZONE)                       # UTC → Ortszeit
dezimal = lambda s: lokal(s).dt.hour + lokal(s).dt.minute / 60           # Uhrzeit als Dezimalzahl (z. B. 21,5)
d = pd.to_datetime(sonne["datum"])
ax[1].fill_between(d, 0, dezimal(sonne["daemmerung_beginn_utc"]), color="midnightblue", alpha=0.5, label="Nacht")
ax[1].fill_between(d, dezimal(sonne["daemmerung_ende_utc"]), 24, color="midnightblue", alpha=0.5)
ax[1].fill_between(d, dezimal(sonne["daemmerung_beginn_utc"]), dezimal(sonne["sonnenaufgang_utc"]), color="orange", alpha=0.6, label="Dämmerung")
ax[1].fill_between(d, dezimal(sonne["sonnenaufgang_utc"]), dezimal(sonne["sonnenuntergang_utc"]), color="gold", alpha=0.4, label="Tag")
ax[1].fill_between(d, dezimal(sonne["sonnenuntergang_utc"]), dezimal(sonne["daemmerung_ende_utc"]), color="orange", alpha=0.6)
ax[1].set(title=f"Tag, Dämmerung und Nacht in Ortszeit ({ORT['name']})", ylabel="Uhrzeit", ylim=(0, 24), yticks=range(0, 25, 3))
ax[1].legend(loc="center right", fontsize=8)
for a in ax:
    a.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))           # x-Achse: Monat/Jahr

plt.tight_layout()
bild_speichern(plt.gcf(), "n3_wetter_sonne.png", dpi=120)      # Grafik speichern
plt.show()

**Ergebnis:**

Außentemperatur plausibel: Jahresgang, Frostperiode Ende Februar/Anfang März 2018 (Tagesmittel bis ca. −12 °C), Hitzesommer 2018 (Tagesmittel bis ca. 28 °C). Tag und Dämmerung verschieben sich über das Jahr: Sonnenuntergang im Juni ca. 21:50 Uhr, im Dezember ca. 15:55 Uhr; die Sommerzeit ist als Sprung im März/Oktober sichtbar.

## 9. Speichern nach silver/cleaned

**Wohin?** `cleaned/<stadt>/wetter/wetter_<stadt>_cleaned__<UTC-Zeitstempel>_UTC.csv` (stündlich) und `cleaned/<stadt>/sonne/sonne_<stadt>_cleaned__<UTC-Zeitstempel>_UTC.csv` (täglich) – jede Ausführung erzeugt eine neue Version; ältere Versionen wandern automatisch in das `_archiv` des Ordners.

**Kontrollwert:** 2 neue Dateien

In [ ]:
# --- Speichern mit UTC-Zeitstempel (neue Version je Lauf) ---
for name_ordner, tabelle in [("wetter", w), ("sonne", sonne)]:
    ordner = f"{CLEANED}/{name_ordner}"                                    # Zielordner
    ordner_anlegen(ordner)                                                 # anlegen, falls nötig
    ziel = f"{ordner}/{name_ordner}_{STADT}_cleaned__{stempel}_UTC.parquet"    # Dateiname mit Zeitstempel
    tabelle.to_parquet(ziel, index=False, storage_options=SO)                  # speichern
    for alt, _ in dateien(ordner, f"{name_ordner}_{STADT}_cleaned__*_UTC.parquet"):  # ältere Versionen ins _archiv des Ordners
        if alt.rsplit("/", 1)[-1] != ziel.rsplit("/", 1)[-1]:
            archivieren(alt)
    print(f"gespeichert: {ziel.replace(WURZEL, '')}  ({len(tabelle):,} Zeilen)")

**Ergebnis:**

2 Dateien gespeichert: Wetter stündlich (21.145 Zeilen) und Sonnenzeiten täglich (881 Tage).

## Zusammenfassung N3

| Datensatz | Zeilen | Inhalt | Quelle |
| :-- | --: | :-- | :-- |
| wetter | 21.145 (stündlich) | Außentemperatur, Niederschlag, Sonnenschein, Wind, Luftfeuchte, Bewölkung, Luftdruck, Taupunkt, Windrichtung, Windböen, Sonnenstrahlung, Wetterlage, Stationen, Sonnenhöhe, Tageszeit | Bright Sky / DWD |
| sonne | 881 (täglich) | Dämmerungsbeginn, Sonnenaufgang, Sonnenuntergang, Dämmerungsende, Tageslänge | NOAA-Berechnung, gegengeprüft mit Open-Meteo |

Befunde: **W1** Außenwerte überwiegend aus Lübeck-Blankensee (13 km), Niederschlag aus Bad Schwartau (2 km) · **W2** Sonnenschein nachts fehlend → 0 gesetzt.

Nächster Schritt: N4 Gold (Stundentabelle), danach N5 Analyse – Flugverhalten nach Monat, Tageszeit (Tag · Dämmerung · Nacht) und Außentemperatur in 5-Grad-Stufen (Pflichtenheft PH-ITS-002, Schritt 3.7).